# CFPB Consumer Complaints Monitor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/cfpb-consumer-complaints-monitor.ipynb)

**What:** Search and monitor the official CFPB Consumer Complaint Database by company, product, issue, state and date. New-since-last-run alerts or per-company trend summaries with response stats. Published fields only, 3-digit ZIP. No API key.

**Why:** run the [`cfpb-consumer-complaints-monitor`](https://apify.com/transparent_meteorite/cfpb-consumer-complaints-monitor) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Only published, de-identified complaint fields are shown (3-digit ZIP, no consumer names).

## 1. Setup

In [ ]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/cfpb-consumer-complaints-monitor) for every option.

In [ ]:
run_input = {   'mode': 'complaints',
    'companies': ['Wells Fargo'],
    'states': ['MI'],
    'lookbackDays': 60,
    'maxResults': 10}

## 3. Run the actor and read the dataset

In [ ]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/cfpb-consumer-complaints-monitor').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

## 4. Tabulate

Only published, de-identified complaint fields are shown (3-digit ZIP, no consumer names).

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["dateReceived", "company", "product", "issue", "state", "zip3", "companyResponse", "timelyResponse", "submittedVia"] if c in df.columns]
df[SHOW]

## 5. A quick analysis

In [ ]:
print(df['product'].value_counts())
print(df['issue'].value_counts().head(5))
df['companyResponse'].value_counts(dropna=False)

## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [cfpb-consumer-complaints-monitor on the Apify Store](https://apify.com/transparent_meteorite/cfpb-consumer-complaints-monitor).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.